# Chapter 18 &mdash; Church Booleans, Pairs, and Selectors

**Concept 5 of the Chapter 18 decomposition:** *Church Booleans, Pairs, and Selectors*

`TRUE` picks its first argument, `FALSE` its second &mdash; so a Boolean <i>is</i> an if-then-else.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-Church-Booleans-And-Pairs/Concept-Church-Booleans-And-Pairs.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$$\textbf{TRUE} = \lambda t.\lambda f.\,t \qquad \textbf{FALSE} = \lambda t.\lambda f.\,f$$

A Boolean is a **selector**: give it two things and it picks one. So `IF` is almost
nothing &mdash; `IF b t f = b t f` &mdash; because the Boolean **is** the conditional.

The connectives follow by selection:
$\textbf{AND} = \lambda p.\lambda q.\,p\,q\,p$,
$\textbf{OR} = \lambda p.\lambda q.\,p\,p\,q$,
$\textbf{NOT} = \lambda p.\,p\,\textbf{FALSE}\,\textbf{TRUE}$.

**Pairs** use the same trick backwards: $\textbf{PAIR}\,a\,b = \lambda s.\,s\,a\,b$
hands both components to a selector. Then $\textbf{FIRST} = \lambda p.\,p\,\textbf{TRUE}$.

And pairs are what make **PRED** possible: carry $(n-1, n)$ along and take the first
component at the end.

## 2. Definitions

### Booleans, pairs and predecessor

In [ ]:
# --- Church encodings, in Python lambdas --------------------------------
# A Church numeral n is the function that applies f to x, n times.
ZERO  = lambda f: lambda x: x
SUCC  = lambda n: lambda f: lambda x: f(n(f)(x))
ADD   = lambda m: lambda n: lambda f: lambda x: m(f)(n(f)(x))
MUL   = lambda m: lambda n: lambda f: m(n(f))
EXP   = lambda m: lambda n: n(m)

def church(n):
    c = ZERO
    for _ in range(n): c = SUCC(c)
    return c

def unchurch(c):
    return c(lambda k: k + 1)(0)

# Booleans: TRUE picks its first argument, FALSE its second -- so a
# Boolean IS an if-then-else.
TRUE  = lambda t: lambda f: t
FALSE = lambda t: lambda f: f
IF    = lambda b: lambda t: lambda f: b(t)(f)
AND   = lambda p: lambda q: p(q)(p)
OR    = lambda p: lambda q: p(p)(q)
NOT   = lambda p: p(FALSE)(TRUE)

def unbool(b): return b(True)(False)

# Pairs and selectors
PAIR   = lambda a: lambda b: lambda s: s(a)(b)
FIRST  = lambda p: p(TRUE)
SECOND = lambda p: p(FALSE)

# Predecessor and zero-test, which recursion needs
ISZERO = lambda n: n(lambda _: FALSE)(TRUE)
SHIFT  = lambda p: PAIR(SECOND(p))(SUCC(SECOND(p)))
PRED   = lambda n: FIRST(n(SHIFT)(PAIR(ZERO)(ZERO)))
SUB    = lambda m: lambda n: n(PRED)(m)
LEQ    = lambda m: lambda n: ISZERO(SUB(m)(n))

## 3. Tests

A Boolean picks one of two things &mdash; that is the whole encoding.

In [ ]:
print("  TRUE  'yes' 'no' =", TRUE('yes')('no'))
print("  FALSE 'yes' 'no' =", FALSE('yes')('no'))
assert TRUE('yes')('no') == 'yes' and FALSE('yes')('no') == 'no'
print("\nIF b t f = b t f -- the Boolean IS the conditional.")
print("  IF TRUE  'a' 'b' =", IF(TRUE)('a')('b'))
print("  IF FALSE 'a' 'b' =", IF(FALSE)('a')('b'))

The connectives, by selection.

In [ ]:
print("%-8s %-8s %-8s %-8s %s" % ("p", "q", "AND", "OR", "NOT p"))
for p, pn in [(TRUE, 'T'), (FALSE, 'F')]:
    for q, qn in [(TRUE, 'T'), (FALSE, 'F')]:
        print("%-8s %-8s %-8s %-8s %s"
              % (pn, qn, unbool(AND(p)(q)), unbool(OR(p)(q)), unbool(NOT(p))))
assert unbool(AND(TRUE)(TRUE)) and not unbool(AND(TRUE)(FALSE))
assert unbool(OR(FALSE)(TRUE)) and not unbool(OR(FALSE)(FALSE))
assert unbool(NOT(TRUE)) is False

De Morgan, in Church Booleans.

In [ ]:
for p in [TRUE, FALSE]:
    for q in [TRUE, FALSE]:
        assert unbool(NOT(AND(p)(q))) == unbool(OR(NOT(p))(NOT(q)))
        assert unbool(NOT(OR(p)(q)))  == unbool(AND(NOT(p))(NOT(q)))
print("both De Morgan laws hold for all four (p, q) combinations")

**Pairs** hand both components to a selector.

In [ ]:
p = PAIR('left')('right')
print("  FIRST  :", FIRST(p))
print("  SECOND :", SECOND(p))
assert FIRST(p) == 'left' and SECOND(p) == 'right'
print("\nPAIR a b = lambda s. s a b.  FIRST just passes TRUE as the selector.")

**ISZERO**, using the fact that $\overline{0}$ never applies its argument.

In [ ]:
for n in range(5):
    print("  ISZERO %d = %s" % (n, unbool(ISZERO(church(n)))))
assert unbool(ISZERO(church(0))) is True
assert all(unbool(ISZERO(church(n))) is False for n in range(1, 5))
print("\nchurch(0) ignores f entirely, so it returns the x -- which is TRUE.")

**PRED**, the hard one, built from pairs.

In [ ]:
for n in range(6):
    print("  PRED %d = %d" % (n, unchurch(PRED(church(n)))))
assert unchurch(PRED(church(0))) == 0
assert all(unchurch(PRED(church(n))) == n - 1 for n in range(1, 6))
print("\nSHIFT carries (a, b) to (b, b+1).  Applying it n times from (0,0)")
print("gives (n-1, n), and FIRST takes the answer.")

And subtraction and comparison follow.

In [ ]:
for m in range(5):
    for n in range(5):
        assert unchurch(SUB(church(m))(church(n))) == max(m - n, 0)
        assert unbool(LEQ(church(m))(church(n))) == (m <= n)
print("SUB (truncated) and LEQ verified for all m, n in 0..4")
print("  SUB 5 2 =", unchurch(SUB(church(5))(church(2))))
print("  LEQ 2 5 =", unbool(LEQ(church(2))(church(5))))

## 4. Exercises


1. Define `XOR` in Church Booleans. Can you do it without `NOT`?
2. Define `SECOND` without using `FALSE`.
3. Why is `SUB` truncated at zero? Could it be otherwise?

In [ ]:
# Your work for the exercises above.